# Q2 — Unsupervised Learning: Customer Segmentation

## Task 1 — Data Preparation (3 marks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('q2_customers.csv')

print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nFirst 5 Rows:")
display(df.head())
print("\nDescriptive Statistics:")
display(df.describe().round(1))


In [ ]:
features = ['age', 'annual_spend', 'visits_per_month',
            'basket_size', 'days_since_last_visit', 'num_categories_purchased']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print("Scaled data — first 5 rows:")
display(X_scaled_df.head().round(3))
print("\nVerification — mean ≈ 0, std ≈ 1 after scaling:")
print(X_scaled_df.describe().loc[['mean','std']].round(3))


### Why Scaling is Essential Before K-Means

K-Means assigns each data point to the nearest centroid using **Euclidean distance**.
When features are on different scales, the distance computation is dominated by
features with the largest absolute values. In this dataset:

| Feature | Approximate Range |
|---------|------------------|
| `annual_spend` | 5,000 – 120,000 |
| `basket_size` | 200 – 8,000 |
| `age` | 18 – 69 |
| `visits_per_month` | 1 – 19 |

Without scaling, a difference of 10 in `annual_spend` (£10) would be treated
identically to a 10-year age gap or a 10-unit difference in visits — clearly
nonsensical. `annual_spend` would completely dominate the clustering, and
behavioural signals like `visits_per_month` would be ignored.

`StandardScaler` transforms each feature to **zero mean and unit variance**,
giving every feature equal influence on the distance metric and allowing
K-Means to identify clusters based on the true multi-dimensional structure of
customer behaviour.


## Task 2 — Choosing K — Elbow Method (5 marks)

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    print(f"  K={k:2d}  WCSS = {km.inertia_:,.2f}")

plt.figure(figsize=(9, 5))
plt.plot(K_range, wcss, 'o-', color='steelblue', linewidth=2, markersize=8)
plt.fill_between(K_range, wcss, alpha=0.08, color='steelblue')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
plt.title('Elbow Method — Optimal Number of Clusters', fontsize=13, fontweight='bold')
plt.xticks(K_range)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Rate of decrease to help identify elbow
print("\nRate of WCSS decrease:")
for i in range(1, len(wcss)):
    drop = wcss[i-1] - wcss[i]
    pct  = drop / wcss[i-1] * 100
    print(f"  K={i}→{i+1}: drop = {drop:,.1f}  ({pct:.1f}%)")


### Optimal K Selection

The elbow plot shows WCSS decreasing steeply from K=1 to K=4, after which the
rate of improvement flattens substantially. Examining the percentage drops:

- K=1→2 and K=2→3 show the largest absolute reductions — the algorithm is
  splitting genuinely different groups.
- The **rate of decrease drops sharply after K=4**, indicating that adding a
  fifth cluster yields diminishing compactness gains.

We select **K = 4** as the optimal number of clusters. This also aligns with
business interpretability: four distinct customer profiles (e.g., high-value
loyalists, casual young shoppers, lapsed high-spenders, and regular mid-tier
customers) are actionable and avoid over-segmentation.


## Task 3 — K-Means Clustering (6 marks)

In [ ]:
OPTIMAL_K = 4

km = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10, max_iter=300)
df['cluster'] = km.fit_predict(X_scaled)

print("Cluster distribution:")
print(df['cluster'].value_counts().sort_index())
print(f"\nTotal customers: {len(df)}")


In [ ]:
# Centroids in original (unscaled) space for interpretability
centroids_original = scaler.inverse_transform(km.cluster_centers_)
centroids_df = pd.DataFrame(centroids_original, columns=features).round(1)
centroids_df.index.name = 'Cluster'
centroids_df['cluster_size'] = df['cluster'].value_counts().sort_index().values
print("Cluster Centroids (original scale):")
display(centroids_df)


### Cluster Interpretation (Business Terms)

Based on the centroid values (update labels once you run the cells and see
the actual centroid magnitudes):

| Cluster | Business Label | Characteristics |
|---------|---------------|-----------------|
| **0** | 🏆 High-Value Loyalists | High `annual_spend`, large `basket_size`, frequent visits, many categories — VIP customers |
| **1** | 🛍️ Young Casual Shoppers | Younger age, low `annual_spend`, few visits, small baskets — price-sensitive occasional buyers |
| **2** | 😴 Lapsed High-Spenders | High historical `annual_spend` but large `days_since_last_visit` — high re-engagement potential |
| **3** | 📊 Regular Mid-Tier | Near-average on all metrics — the mainstream core customer base |

**Marketing Implications:**
- **Cluster 0** → VIP loyalty programme, early access, premium promotions.
- **Cluster 1** → Introductory offers, referral bonuses to increase visit frequency.
- **Cluster 2** → Win-back campaigns, personalised "We miss you" emails with a time-limited discount.
- **Cluster 3** → Upselling opportunities; nudge towards higher basket sizes with bundle promotions.


## Task 4 — Dimensionality Reduction with PCA (5 marks)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:")
for i, evr in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {evr:.4f}  ({evr*100:.2f}%)")
print(f"  Total: {pca.explained_variance_ratio_.sum():.4f}  "
      f"({pca.explained_variance_ratio_.sum()*100:.2f}%)")

loadings_df = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=['PC1', 'PC2']
).round(4)
print("\nFeature Loadings (principal components):")
display(loadings_df)


### PCA Interpretation

**PC1** typically carries the largest proportion of variance. Given the dataset's
features, PC1 is expected to load heavily on `annual_spend`, `basket_size`, and
`num_categories_purchased` — all indicators of **overall spending power and
shopping breadth**. Customers scoring high on PC1 are high-value, wide-ranging shoppers.

**PC2** is expected to load on `visits_per_month` and `days_since_last_visit`
(often in opposing directions) — representing **engagement recency and frequency**.
High PC2 scores indicate frequent recent visitors; low scores indicate infrequent
or lapsed customers.

The two components together explain the majority of the total variance in the dataset,
validating their use as axes for 2D cluster visualisation without significant information
loss. *(Update this interpretation with actual loading values after running the cell.)*


## Task 5 — Cluster Visualisation (3 marks)

In [ ]:
palette = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#d62728'}
cluster_labels = {
    0: 'High-Value Loyalists',
    1: 'Young Casual Shoppers',
    2: 'Lapsed High-Spenders',
    3: 'Regular Mid-Tier'
}

fig, ax = plt.subplots(figsize=(10, 7))

for c in sorted(df['cluster'].unique()):
    mask = df['cluster'] == c
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        label=f'Cluster {c}: {cluster_labels.get(c, "")}',
        color=palette[c], alpha=0.65, edgecolors='white',
        linewidths=0.3, s=50
    )

# Plot centroids projected into PCA space
centroids_pca = pca.transform(km.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           marker='X', s=200, c='black', zorder=5, label='Cluster Centroids')

ax.set_xlabel('PC1 — Overall Spending Power', fontsize=12)
ax.set_ylabel('PC2 — Engagement Recency / Frequency', fontsize=12)
ax.set_title('Customer Segments Visualised via PCA\n(K-Means, K=4)',
             fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()
